[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pradeepvaka/llm-inference-90day/blob/master/notebooks/day03-attention-math-by-hand.ipynb)
# Day 3 — Attention Math, by Hand
Implement scaled dot-product attention in NumPy, verify it against PyTorch, and measure the O(n^2) cost numerically. CPU-only.

In [ ]:
!pip install -q numpy matplotlib torch --index-url https://download.pytorch.org/whl/cpu
# Expected: installs quietly, no output

## 1. The worked example
Raw QK^T scores for d_k=4, n=3, then causal mask (-inf above diagonal), scale by sqrt(4)=2, softmax.

In [ ]:
import numpy as np

scores = np.array([[4., 1, 0],
                   [2, 5, 1],
                   [1, 2, 6]])
d_k = 4

def causal_attention_from_scores(scores, V, d_k):
    n = scores.shape[0]
    masked = scores / np.sqrt(d_k)
    masked[np.triu_indices(n, k=1)] = -np.inf   # causal: no peeking at the future
    e = np.exp(masked - masked.max(axis=1, keepdims=True))
    weights = e / e.sum(axis=1, keepdims=True)
    return weights, weights @ V

V = np.array([[1., 0, 2, 1],
              [0, 1, 1, 0],
              [2, 2, 0, 1]])
W, out = causal_attention_from_scores(scores, V, d_k)
print("weights:\n", np.round(W, 3))
print("row0 output:", np.round(out[0], 3))
# Expected:
# weights:
#  [[0.818 0.182 0.   ]
#   [0.182 0.818 0.   ]
#   [0.067 0.111 0.821]]
# row0 output: [0.818 0.182 1.818 0.818]

## 2. Verify against PyTorch
`F.scaled_dot_product_attention` with `is_causal=True` must agree to 1e-5.

In [ ]:
import torch, torch.nn.functional as F

torch.manual_seed(0)
n, d_k = 64, 64
Q = torch.randn(1, 4, n, d_k); K = torch.randn(1, 4, n, d_k); Vv = torch.randn(1, 4, n, d_k)

# reference: our NumPy math, head by head
def numpy_ref(q, k, v):
    s = q @ k.T / np.sqrt(d_k)
    s[np.triu_indices(n, k=1)] = -np.inf
    e = np.exp(s - s.max(axis=1, keepdims=True)); w = e / e.sum(axis=1, keepdims=True)
    return w @ v

ref = np.stack([numpy_ref(Q[0,h].numpy(), K[0,h].numpy(), Vv[0,h].numpy()) for h in range(4)])
got = F.scaled_dot_product_attention(Q, K, Vv, is_causal=True).numpy()
print("max abs diff:", np.abs(ref - got).max())
assert np.allclose(ref, got, atol=1e-5)
print("MATCH: NumPy == torch sdpa (causal)")
# Expected: max abs diff: ~1e-7, then MATCH

## 3. Why the 1/sqrt(d_k) scaling matters
Random q,k with unit variance: watch weights saturate to one-hot without scaling at d_k=128.

In [ ]:
for dk, scaled in [(128, False), (128, True)]:
    q = np.random.randn(dk); K = np.random.randn(8, dk)
    logits = (q @ K.T) / (np.sqrt(dk) if scaled else 1.0)
    e = np.exp(logits - logits.max()); w = e / e.sum()
    print(f"d_k={dk} scaled={scaled}: max weight={w.max():.3f}, entropy={-(w*np.log(w+1e-12)).sum():.2f} nats")
# Expected: unscaled -> max weight ~0.99 (one-hot, saturated); scaled -> max weight ~0.3-0.5, entropy higher

## 4. Measure the O(n^2) blowup
Time attention for n in {512..8192} (CPU); fit the log-log slope — expect ~2.0.

In [ ]:
import time, math
import matplotlib.pyplot as plt

def bench(n, dk=64, trials=5):
    q = torch.randn(1, 1, n, dk); k = torch.randn(1, 1, n, dk); v = torch.randn(1, 1, n, dk)
    F.scaled_dot_product_attention(q, k, v, is_causal=True)  # warmup
    t0 = time.perf_counter()
    for _ in range(trials):
        F.scaled_dot_product_attention(q, k, v, is_causal=True)
    return (time.perf_counter() - t0) / trials

ns = [512, 1024, 2048, 4096, 8192]
ts = [bench(n) for n in ns]
for n, t in zip(ns, ts): print(f"n={n:5d}  {t*1000:8.1f} ms")
# Expected (CPU, rough): ~2, ~8, ~35, ~150, ~700 ms — each doubling ~4x the time

lx, ly = np.log(ns), np.log(ts)
slope = np.polyfit(lx, ly, 1)[0]
print(f"log-log slope: {slope:.2f}  (quadratic => 2.0)")
plt.loglog(ns, ts, "o-"); plt.xlabel("n (tokens)"); plt.ylabel("seconds")
plt.title(f"Attention time vs n (slope={slope:.2f})"); plt.grid(True); plt.show()

## 5. Score-matrix bytes: the formula
`n^2 x heads x bytes_per_element` per layer. Confirm 4.0 GiB at n=8192, 32 heads, fp16.

In [ ]:
def score_bytes(n, n_heads=32, bytes_per_elem=2):
    return n**2 * n_heads * bytes_per_elem

for n in [1024, 4096, 8192, 32768]:
    gib = score_bytes(n) / 2**30
    print(f"n={n:6d}: {gib:8.2f} GiB per layer   ({gib*32:8.1f} GiB over 32 layers)")
assert abs(score_bytes(8192) / 2**30 - 4.0) < 1e-9
print("CONFIRMED: 4.0 GiB per layer at n=8192")
# Expected: 1024 -> 0.06 GiB; 4096 -> 1.00 GiB; 8192 -> 4.00 GiB; 32768 -> 64.00 GiB

## Wrap-up
- Attention = softmax(QK^T/sqrt(d_k)) V: a soft dictionary lookup over values.
- sqrt(d_k) scaling keeps logits at unit variance; without it softmax saturates and gradients die.
- The score matrix is n^2 per head: 1 GiB/layer at n=4096 (32 heads, fp16) — the quadratic wall.
- Tomorrow (Day 4): split into heads (MHA), count the params, and learn RoPE.